In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from sklearn import datasets
from sklearn import linear_model

from sklearn.model_selection import train_test_split
#from sklearn.cross_validation import train_test_split
from sklearn.preprocessing import PolynomialFeatures 

from itertools import combinations 

#### diabetes is another dataset provided by sklearn.datasets

In [ ]:
diabetes = datasets.load_diabetes()

In [ ]:
df = pd.DataFrame(diabetes.data)    #since above codes is for dataframe, we use pandas to change the data into dataframe
df.columns = diabetes.feature_names #assign the feature names to the dataframe
df['target'] = diabetes.target

df = df[['s1', 's2', 's3', 's5', 'target']]

In [ ]:
df.head()

In [ ]:
sns.pairplot(pd.DataFrame(df)) #plot pairplot
plt.show()

In [ ]:
def correlation_heatmap(data):
    _ , ax = plt.subplots(figsize =(14, 12)) #Only ax is used in following code, we do not assign value to the other variable
    colormap = sns.diverging_palette(220, 10, as_cmap = True)
    
    sns.heatmap(
        data.corr(), 
        cmap = colormap,
        square=True, 
        cbar_kws={'shrink':.9 }, 
        ax=ax,
        annot=True, 
        linewidths=0.1,vmax=1.0, linecolor='white',
        annot_kws={'fontsize':12 }
    )
    
    plt.title('Pearson Correlation of Features', y=1.05, size=15)
    plt.show()
correlation_heatmap(df) #Input train_df to the function, and check the correlation of all variables

In [ ]:
#split the data into training set and testing set, and set seed = 0
train, test = train_test_split(df, test_size = 0.1, random_state = 0)
print('Size of the train dataset is ', train.shape)
print('Size of the test dataset is ', test.shape)

In [ ]:
x_train = train.iloc[:, :-1]
y_train = train['target']
x_test = test.iloc[:, :-1]
y_test = test['target']

In [ ]:
#fit ordinary linear regression
lr = linear_model.LinearRegression().fit(x_train, y_train)
print('Coefficients: \n', lr.coef_)

# Explained variance score: 1 is perfect prediction
print('R^2: %.2f' % lr.score(x_train, y_train))

In [ ]:
def AIC(train_data, train_label):
    from sklearn import linear_model
    linear_model = linear_model.LinearRegression().fit(train_data, train_label)
    predict_label = linear_model.predict(train_data)
    degree_freedom = train_data.shape[1]
    sample_size = train_data.shape[0]
    SSE = np.dot((train_label - predict_label), (train_label - predict_label))
    return sample_size * np.log(SSE/sample_size) + 2 * degree_freedom

In [ ]:
def BIC(train_data, train_label):
    from sklearn import linear_model
    linear_model = linear_model.LinearRegression().fit(train_data, train_label)
    predict_label = linear_model.predict(train_data)
    degree_freedom = train_data.shape[1]
    sample_size = train_data.shape[0]
    SSE = np.dot((train_label - predict_label), (train_label - predict_label))
    return sample_size * np.log(SSE/sample_size) + np.log(sample_size) * degree_freedom

In [ ]:
feature_names = list(x_train.columns.values)
min_AIC = []
min_AIC.append(y_train.var())
plt.scatter(1, y_train.var()) #when there is only the intercept term, plot variance
for i in range(1, 5):
    names = list(combinations(feature_names, i)) #generate the combination of i feature names and convert it to a list
    print(names)
    aic_list = []    #define a list to store AIC values
    for each_element in names:
        aic_list.append(AIC(x_train[list(each_element)], y_train)) #append BIC values to the list
    min_AIC.append(np.min(aic_list))
    print(aic_list)
    for each_value in aic_list:
        plt.scatter(x = i + 1, y = each_value)   #plot AIC values
    
plt.plot([1, 2, 3, 4, 5], min_AIC)
plt.title('AIC') 
plt.xlabel('Degree of Freedom')
plt.ylabel('AIC Value')
plt.show()

In [ ]:
print('The minimum AICs are:',min_AIC)

In [ ]:
feature_names = list(x_train.columns.values)
min_BIC = []
min_BIC.append(y_train.var())
plt.scatter(1, y_train.var()) #when there is only the intercept term, plot variance
for i in range(1, 5):
    names = list(combinations(feature_names, i)) #generate the combination of i feature names and convert it to a list
    print(names)
    bic_list = []  #define a list to store BIC values
    for each_element in names:
        bic_list.append(BIC(x_train[list(each_element)], y_train))  #append BIC values to the list
    print(bic_list)
    min_BIC.append(np.min(bic_list))
    for each_value in bic_list:
        plt.scatter(x = i + 1, y = each_value) #plot BIC values
plt.plot([1, 2, 3, 4, 5], min_BIC)
plt.title('BIC')
plt.xlabel('Degree of Freedom')
plt.ylabel('BIC Value')
plt.show()

In [ ]:
print('The minimum BICs are:',min_BIC)

In [ ]:
n_alphas_ridge = 200
#sample 200 samples from -5 to 3 in an equal space in log space that is 10^(-5) to 10^(-3)
alphas_ridge = np.logspace(-5, 3, n_alphas_ridge)

#fit ridge regression with penalty of alphas_ridge
model = linear_model.RidgeCV(alphas=alphas_ridge, store_cv_values=True).fit(x_train, y_train)
print('The penalty should be: %f'%model.alpha_)

In [ ]:
n_alphas_ridge = 200
#sample 200 samples from -5 to 3 in an equal space in log space that is 10^(-5) to 10^(-3)
alphas_ridge = np.logspace(-5, 3, n_alphas_ridge)

coefs_ridge = []
for a_ridge in alphas_ridge:
    ridge = linear_model.Ridge(alpha=a_ridge, fit_intercept=False)
    ridge.fit(x_train, y_train)
    
    #append coefficient to the list 'coefs_ridge'
    coefs_ridge.append(ridge.coef_)

# #############################################################################
# Display results

ax = plt.gca()

ax.plot(alphas_ridge, coefs_ridge)
#vertical dashed line with black color, indicating the best alpha value
ax.axvline(model.alpha_, color = 'k', linestyle='--')
ax.set_xscale('log')
# reverse axis
ax.set_xlim(ax.get_xlim()[::-1])
plt.xlabel('alpha')
plt.ylabel('weights')
plt.title('Ridge coefficients as a function of the regularization')
plt.axis('tight')
plt.show()

In [ ]:
print("Computing regularization path using the Lars lasso...")

#10 fold cross validation
#fit lasso regression
model = linear_model.LassoCV(cv=10).fit(x_train, y_train)

# Display results
m_log_alphas = -np.log10(model.alphas_)

plt.figure()
plt.plot(m_log_alphas, model.mse_path_, ':')
plt.plot(m_log_alphas, model.mse_path_.mean(axis=-1), 'k',label='Average across the folds', linewidth=2)
#vertical dashed line with black color, indicating the best alpha value
plt.axvline(-np.log10(model.alpha_), linestyle='--', color='k',label='alpha: CV estimate')

plt.legend()

plt.xlabel('-log(alpha)')
plt.ylabel('Mean square error')
plt.title('Mean square error on each fold')
plt.axis('tight')
plt.show()

In [ ]:
n_alphas_lasso = 200
#sample 200 samples from -5 to 3 in an equal space in log space that is 10^(-5) to 10^(-3)
alphas_lasso = np.logspace(-5, 3, n_alphas_lasso) 

coefs_lasso = []
for a_lasso in alphas_lasso:
    #fit lasso regression with penalty of a_lasso
    lasso = linear_model.Lasso(alpha=a_lasso, fit_intercept=False)
    lasso.fit(x_train, y_train)
    coefs_lasso.append(lasso.coef_)

# #############################################################################
# Display results

ax = plt.gca()

ax.plot(alphas_lasso, coefs_lasso)
ax.set_xscale('log')
# reverse axis
ax.set_xlim(ax.get_xlim()[::-1])  
#vertical dashed line with black color, indicating the best alpha value
ax.axvline(model.alpha_, color = 'k', linestyle='--')
plt.xlabel('alpha')
plt.ylabel('weights')
plt.title('Ridge coefficients as a function of the regularization')
plt.axis('tight')
plt.show()

In [ ]:
model_aic = linear_model.LassoLarsIC(criterion='aic') #Lasso model fit with Lars using AIC 
model_aic.fit(x_train, y_train) 
alpha_aic_ = model_aic.alpha_ #the alpha parameter chosen by AIC
coef_aic_ = model_aic.coef_ #the coefficients estimated by BIC

model_bic = linear_model.LassoLarsIC(criterion='bic') #Lasso model fit with Lars using BIC 
model_bic.fit(x_train, y_train)
alpha_bic_ = model_bic.alpha_ #the alpha parameter chosen by BIC
coef_bic_ = model_bic.coef_ #the coefficients estimated by BIC


#Input is model, name, and color
def plot_ic_criterion(model, name, color):
    alpha_ = model.alpha_
    alphas_ = model.alphas_
    criterion_ = model.criterion_
    print('%s coefficient is: '%name, model.coef_)
    plt.plot(-np.log10(alphas_), criterion_, '--', color=color,linewidth=3, label='%s criterion' % name)
    plt.axvline(-np.log10(alpha_), color=color, linewidth=3, label='alpha: %s estimate' % name)
    plt.xlabel('-log(alpha)')
    plt.ylabel('criterion')

plt.figure()
plot_ic_criterion(model_aic, 'AIC', 'b')
plot_ic_criterion(model_bic, 'BIC', 'r')
plt.legend()
plt.title('Information-criterion for model selection')
plt.show()
print('BIC alpha is: %f'%alpha_bic_)
print('AIC alpha is: %f'%alpha_aic_)

In [ ]:
#transform the model to model with 2 degrees
poly = PolynomialFeatures(2)
x_poly = poly.fit_transform(x_train)
poly.fit(x_train, y_train)
#fit linear regression with transformed data
pr = linear_model.LinearRegression().fit(x_poly, y_train)
print('Coefficients: \n', pr.coef_)

In [ ]:
# Explained variance score: 1 is perfect prediction
print('R^2: %.2f' % pr.score(x_poly, y_train))